# 코드 생성기

목표: Frontier 모델을 사용하여 Python 코드로부터 고성능 C++ 코드를 생성합니다


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">알림: 최신 코드 가져오기</h2>
            <span style="color:#f71;">저는 지속적으로 이 실습을 개선하고 더 많은 예제와 연습문제를 추가하고 있습니다.
            매주 초에 최신 코드가 있는지 확인하는 것이 좋습니다.<br/>
            먼저 <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull을 실행하고 필요에 따라 변경사항을 병합하세요</a>. 문제가 있으신가요? ChatGPT에 병합 방법을 물어보거나 저에게 연락하세요!<br/><br/>
            코드를 가져온 후, llm_engineering 디렉토리에서 Cursor 터미널에 다음을 실행하세요:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">중요 참고사항</h1>
            <span style="color:#900;">
            이 실습에서는 GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4와 같이 약간 더 높은 가격의 고성능 모델을 사용합니다. 비용은 여전히 낮지만, 비용을 최소화하고 싶으시다면 gpt-5-nano와 같은 저비용 모델을 선택하세요.
            </span>
        </td>
    </tr>
</table>

In [18]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [19]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AI
Grok API Key not set (and this is optional)


In [20]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)

In [21]:
OPENAI_MODEL = "gpt-4o-mini"
CLAUDE_MODEL = "claude-3-5-haiku-latest"
GROK_MODEL = "grok-4-fast-non-reasoning"
GEMINI_MODEL = "gemini-2.5-flash-lite"

# 고성능 모델을 원하면 아래 주석을 해제하세요:

# OPENAI_MODEL = "gpt-5"
# CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
# GROK_MODEL = "grok-4"
# GEMINI_MODEL = "gemini-2.5-pro"

## 주의사항:

여기서는 Python 코드를 여러분의 머신에서 실행 가능한 효율적이고 최적화된 C++ 코드로 변환하는 솔루션을 작성할 것입니다. 이 코드는 네이티브 머신 코드로 컴파일되어 실행될 수 있습니다.

직접 코드를 실행하지 않아도 괜찮습니다 - 이것이 이 연습의 핵심은 아닙니다!

하지만 (만족감을 위해!) 원하신다면 여기에 단계를 포함시켰습니다. 완전히 선택사항입니다!

대안으로, C++ 코드를 실행할 수 있는 웹사이트도 소개해 드리겠습니다.

In [22]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': 'AMD Ryzen 7 3700X 8-Core Processor',
  'cores_logical': 16,
  'cores_physical': 8,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (tdm64-1) 10.3.0',
   'g++': 'g++.EXE (tdm64-1) 10.3.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [23]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

Based on the system information you've provided, it appears that you already have the `g++` compiler installed, which is part of the MinGW toolchain on your Windows system. This means you do not need to install anything further to compile your C++ code.

### Step-by-step Instructions to Compile and Execute `main.cpp`

**1. Compile the C++ File**

To compile the `main.cpp` file, you can use the following `g++` command. Here’s the exact code to use in your Python script for compiling:
```python
compile_command = ["g++", "-O3", "main.cpp", "-o", "main.exe"]
```
In this command:
- `-O3` is a compiler option that enables optimizations for faster runtime performance.
- `main.cpp` is your source file.
- `-o main.exe` specifies the name of the output executable file.

**2. Run the Compiled Executable**

To run the compiled executable, you can specify the following:
```python
run_command = ["main.exe"]
```

### Complete Python Code Example

Here’s the complete Python code to compile and run the C++ program:

```python
import subprocess

compile_command = ["g++", "-O3", "main.cpp", "-o", "main.exe"]
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

run_command = ["main.exe"]
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

# Assuming you want to return the output of the program
print(run_result.stdout)
```

### Summary

You already have everything set up to compile and run your C++ code using `g++` on Windows. The provided Python code will compile `main.cpp` with optimizations and then execute the resulting `main.exe`. Just make sure that `main.cpp` is in the same directory as your script or provide an appropriate path to it.

In [24]:
compile_command = ["g++", "-O3", "-march=native", "-flto", "-DNDEBUG", "-std=c++17", "main.cpp", "-o", "main.exe"] 
run_command = [".\main.exe"]

## 설치가 필요한 경우

원하신다면 GPT의 지시를 따르세요! 그런 다음 설정이 완료되었는지 확인하기 위해 분석을 다시 실행하세요 (노트북을 재시작해야 할 수도 있습니다).

이제 코드를 컴파일하는 명령과 실행하는 명령을 갖추게 되었습니다!

아래 셀에 입력하세요:

In [25]:
import platform
import shutil

# Windows: use MinGW (g++) to compile generated main.cpp.
# This cell is intentionally simple; if g++ is missing, we fail loudly with a clear hint.
if platform.system() == "Windows":
    from pathlib import Path
    gpp = shutil.which("g++") or r"C:\\TDM-GCC-64\\bin\\g++.exe"
    if not Path(gpp).exists():
        raise RuntimeError(
            "g++ not found. Install TDM-GCC (MinGW-w64) via: winget install -e --id JMEubank.TDM-GCC, then restart the terminal."
        )
    compile_command = [
        gpp,
        "-O3",
        "-march=native",
        "-flto",
        "-DNDEBUG",
        "-std=c++20",
        "main.cpp",
        "-o",
        "main.exe",
        "-static-libstdc++",
        "-static-libgcc",
    ]
    run_command = [".\\main.exe"]
else:
    # Non-Windows fallback: prefer clang++ if available.
    if shutil.which("clang++"):
        compile_command = [
            "clang++",
            "-std=c++17",
            "-Ofast",
            "-mcpu=native",
            "-flto=thin",
            "-fvisibility=hidden",
            "-DNDEBUG",
            "main.cpp",
            "-o",
            "main",
        ]
        run_command = ["./main"]
    elif shutil.which("g++"):
        gpp = shutil.which("g++")
        compile_command = [
            gpp,
            "-O3",
            "-march=native",
            "-flto",
            "-DNDEBUG",
            "-std=c++20",
            "main.cpp",
            "-o",
            "main",
        ]
        run_command = ["./main"]
    else:
        raise RuntimeError("No C++ compiler found (need g++ or clang++).")

## 이제 본격적으로 시작해봅시다

In [26]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [27]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [28]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [29]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [30]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [31]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [ ]:
run_python(pi)

In [32]:
port(openai, OPENAI_MODEL, pi)

BadRequestError: Error code: 400 - {'error': {'message': 'Unrecognized request argument supplied: reasoning_effort', 'type': 'invalid_request_error', 'param': None, 'code': None}}

# C++ 컴파일 및 실행

다음 셀에는 GPT의 지시에 따라 C++ 파일을 컴파일하는 명령이 포함되어 있습니다.

다시 말씀드리지만, 원하지 않으신다면 이 단계를 반드시 수행할 필요는 없습니다!

또는 대안으로: 학생 Sandeep K.G.가 Python과 C++ 코드를 온라인에서 테스트할 수 있다고 알려주셨습니다. Sandeep 감사합니다!  
> 정확한 비교는 아니지만 성능 차이를 파악할 수 있습니다.  
> 예를 들어 여기에서: https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [ ]:
compile_and_run()

In [ ]:
19.178207/0.082168

## 자, 다른 경쟁자들도 시도해봅시다!

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()

In [ ]:
port(grok, GROK_MODEL, pi)
compile_and_run()

In [ ]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


In [ ]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")